# Limpeza das bases de dados
O objetivo desse notebook é identificar e tratar inconsistências nas bases antes de realizar quaisquer análises. 
Está detalhado todos os problemas encontrados e a decisão tomada para tratamento destes. 

# Bases analisadas
- pedidos.csv = registro de todos os pedidos 
- clientes.csv = cadastro de clientes 
- produtos.csv = catálogo de produtos 
- itens_pedido.csv = itens individuais de cada pedido
- avaliacoes.csv = avaliações de clientes 
- tickets_suporte.csv = chamados de suporte 

In [1]:

# Imports e configurações globais

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path


# Configurações de visualização 
# Estilo padrão para os gráficos do projeto
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.family": "sans-serif",
})

# Configurações do Pandas 
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Caminhos
# O Path() garante compatibilidade entre Windows, Mac e Linux
caminho_base = Path("../base_dados")


#  Carregamento das Bases
 Foi aplicado os tipos de dados corretos, como datas como datetime, para evitar retrabalho nas etapas seguintes. 

In [2]:

# Usei parse_dates para que o pandas interprete colunas de data como datetime64, e não como string.

df_pedidos = pd.read_csv(
    caminho_base / "pedidos.csv",
    parse_dates=["data_pedido"]
)

df_clientes = pd.read_csv(
    caminho_base / "clientes.csv",
    parse_dates=["data_cadastro"]
)

df_produtos = pd.read_csv(caminho_base / "produtos.csv")

df_itens = pd.read_csv(caminho_base / "itens_pedido.csv")

df_avaliacoes = pd.read_csv(
    caminho_base / "avaliacoes.csv",
    parse_dates=["data_avaliacao"]
)

df_tickets = pd.read_csv(
    caminho_base / "tickets_suporte.csv",
    parse_dates=["data_abertura", "data_resolucao"]
)

# Dicionário central
bases = {
    "pedidos":        df_pedidos,
    "clientes":       df_clientes,
    "produtos":       df_produtos,
    "itens_pedido":   df_itens,
    "avaliacoes":     df_avaliacoes,
    "tickets_suporte": df_tickets,
}

#print("Bases carregadas com sucesso!\n")
#for nome, df in bases.items():
#    print(f" {nome:<20} = {df.shape[0]:>6} linhas e {df.shape[1]} colunas")


# Diagnóstico dos dados
Antes de tomar qualquer decisão de tratamento, fiz um diagnóstico de todas as bases.  

In [5]:


def auditar_dataframe(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    """
    Gera relatório para um DataFrame.

    Retorna uma tabela com:
    - Tipo de dado de cada coluna
    - Quantidade e percentual de valores nulos
    - Quantidade de valores únicos
    - Presença de duplicatas na chave primária 'id'
    """
    relatorio = pd.DataFrame({
        "tipo_dado":      df.dtypes.astype(str),
        "nulos_qtd":      df.isnull().sum(),
        "nulos_pct":      (df.isnull().sum() / len(df) * 100).round(2),
        "valores_unicos": df.nunique(),
    })

    duplicatas = df.duplicated().sum() if "id" not in df.columns else df["id"].duplicated().sum()

    print(f" {nome.upper()}")
    print(f"  Linhas      : {df.shape[0]:,}")
    print(f"  Colunas     : {df.shape[1]}")
    print(f"  Duplicatas  : {duplicatas}")
    display(relatorio.style.background_gradient(
        subset=["nulos_pct"], cmap="Reds"
    ))
    print()
    return relatorio


relatorios = {}
for nome, df in bases.items():
    relatorios[nome] = auditar_dataframe(df, nome)

 PEDIDOS
  Linhas      : 15,000
  Colunas     : 7
  Duplicatas  : 0


,tipo_dado,nulos_qtd,nulos_pct,valores_unicos
id,int64,0,0.000000,15000
cliente_id,int64,0,0.000000,2977
data_pedido,datetime64[ns],0,0.000000,731
status,object,0,0.000000,4
valor_total,float64,79,0.530000,14633
canal_venda,object,0,0.000000,3
cupom_desconto,object,0,0.000000,2



 CLIENTES
  Linhas      : 3,000
  Colunas     : 8
  Duplicatas  : 0


,tipo_dado,nulos_qtd,nulos_pct,valores_unicos
id,int64,0,0.000000,3000
nome,object,0,0.000000,1519
email,object,0,0.000000,3000
cidade,object,0,0.000000,40
estado,object,0,0.000000,23
data_cadastro,datetime64[ns],0,0.000000,713
segmento,object,0,0.000000,2
canal_aquisicao,object,0,0.000000,4



 PRODUTOS
  Linhas      : 200
  Colunas     : 7
  Duplicatas  : 0


...

,tipo_dado,nulos_qtd,nulos_pct,valores_unicos
id,int64,0,0.000000,36740
pedido_id,int64,0,0.000000,15000
produto_id,int64,0,0.000000,200
quantidade,int64,0,0.000000,5
preco_praticado,float64,0,0.000000,5550
desconto_aplicado,float64,301,0.820000,31



 AVALIACOES
  Linhas      : 8,000
  Colunas     : 7
  Duplicatas  : 0


,tipo_dado,nulos_qtd,nulos_pct,valores_unicos
id,int64,0,0.000000,8000
pedido_id,int64,0,0.000000,8000
produto_id,int64,0,0.000000,200
cliente_id,int64,0,0.000000,2791
nota,int64,0,0.000000,5
comentario,object,1679,20.990000,13
data_avaliacao,datetime64[ns],0,0.000000,724



 TICKETS_SUPORTE
  Linhas      : 4,000
  Colunas     : 7
  Duplicatas  : 0


,tipo_dado,nulos_qtd,nulos_pct,valores_unicos
id,int64,0,0.000000,4000
pedido_id,int64,0,0.000000,4000
cliente_id,int64,0,0.000000,2201
categoria_problema,object,0,0.000000,5
data_abertura,datetime64[ns],0,0.000000,720
data_resolucao,datetime64[ns],1653,41.320000,688
status,object,0,0.000000,3


# Critérios de decisão utilizados:

Temos 79 nulos em valor_total e decidi preencher os valores cruzando o Id do pedido para obter o valor real. 
Nulos em campo opcional, como avaliação, decidi manter como NaN, pois a ausência de comentário é válida
Nulos em campo operacionalmente consistente, decidi manter nulo. Por exemplo, um ticket aberto sem data de resolução é o comportamento esperado 
Nulos que representam clientes sem pedidos, também decidi manter como nulo, pois podem ser dados estratégicos.

# Arrumando a tabela itens_pedido: 

In [6]:
# Temos 301 nulos em desconto_aplicado e decidi atribuir 0 para esses campos, pois a explicação mais 
# razoável é que nenhum desconto foi aplicado nesses casos. 
# Caso fosse utilizado mediana para preencher esses campos, formaríamos descontos fictícios em transações
# que possivelmente foram a preço cheio. 

nulos_antes = df_itens["desconto_aplicado"].isnull().sum()
df_itens["desconto_aplicado"] = df_itens["desconto_aplicado"].fillna(0.0)
nulos_depois = df_itens["desconto_aplicado"].isnull().sum()

# print(f"  Shape original: {df_itens.shape}")
# print(f"\n desconto_aplicado: Nulos =  {nulos_antes} → {nulos_depois}")

# Criei uma coluna que representa o valor real pago após o desconto para responder algumas perguntas da questão 2. 

df_itens["valor_liquido_item"] = (
    df_itens["preco_praticado"]
    * df_itens["quantidade"]
    * (1 - df_itens["desconto_aplicado"])
).round(2)

# print(f"\n Nova coluna criada: 'valor_liquido_item'")

# print(f"\n  Shape final: {df_itens.shape}")
# print("\nItens_pedido está arrumado")
# df_itens.head(3)

# Arrumando a tabela Pedidos: 

In [7]:
# Temos 79 nulos em valor_total. Preenchi os valores nulos cruzando o Id do pedido para obter o valor real. 
nulos_antes_pedidos = df_pedidos["valor_total"].isnull().sum()

# Calculando a receita real agrupando os itens de cada pedido
receita_por_pedido = df_itens.groupby('pedido_id')['valor_liquido_item'].sum()

df_pedidos = df_pedidos.set_index('id')
df_pedidos['valor_total'] = df_pedidos['valor_total'].combine_first(receita_por_pedido)
df_pedidos = df_pedidos.reset_index()

nulos_depois_pedidos = df_pedidos["valor_total"].isnull().sum()

# Arrumando a coluna cupom 
df_pedidos["cupom_desconto"] = df_pedidos["cupom_desconto"].map({"sim": True, "não": False})

print(f"Coluna valor_total: Nulos corrigidos de {nulos_antes_pedidos} para {nulos_depois_pedidos} via soma dos itens.")

Coluna valor_total: Nulos corrigidos de 79 para 0 via soma dos itens.


# Arrumando a tabela Avaliações

In [8]:

# Temos 1.679 nulos em comentario. Decidi manter como NaN, pois a ausência de comentários é um dado válido. 
# O cliente avaliou de maneira numérica, mas optou por não escrever, o que é válido. 
# Se substituíssemos o comentário vazio por um escrito "sem comentário", poderíamos criar análises futuras. 

pct_sem_comentario = df_avaliacoes["comentario"].isnull().mean() * 100

# Criei uma coluna booleana para facilitar análises de engajamento
df_avaliacoes["tem_comentario"] = df_avaliacoes["comentario"].notna()

# Criei categorias para a experiência do cliente. 
# 1–2 = negativa | 3 = neutra | 4–5 = positiva
df_avaliacoes["sentimento"] = pd.cut(
    df_avaliacoes["nota"],
    bins=[0, 2, 3, 5],
    labels=["negativo", "neutro", "positivo"]
)

# print(f"Shape original: {df_avaliacoes.shape}")
# print(f"\nNova coluna: 'tem_comentario' (bool)")
# print(f"Nova coluna: 'sentimento' (negativo/neutro/positivo)")
# print(f"\nDistribuição de experiência:")
# print(df_avaliacoes["sentimento"].value_counts())

# print(f"\n  Shape final: {df_avaliacoes.shape}")
# print("\nAvaliações foi arrumado")
# df_avaliacoes.head(3)

# Arrumando Tickets de suporte 

In [9]:
# Temos 1.653 nulos em data_resolucao e decidi manter como NaN, pois a análise de consistência feita
# revelou que todos os tickets que estão com o status 'aberto' ou 'escalado' tem data de resolução nula. 
# Desa forma, tratar esses dados apagaria informações reais, pois nenhum ticket 'resolvido' está com a data nula. 


tickets_abertos = df_tickets[df_tickets["status"] != "resolvido"].shape[0]
# print(f"Shape original: {df_tickets.shape}")
# print(f"\ndata_resolucao {df_tickets['data_resolucao'].isnull().sum()} nulos")
# print(f"Tickets abertos/escalados: {tickets_abertos} → Consistente")


# Calculei o tempo de resolução em dias (somente para tickets já resolvidos)
df_tickets["tempo_resolucao_dias"] = (
    df_tickets["data_resolucao"] - df_tickets["data_abertura"]
).dt.days

# print(f"\n Nova coluna: 'tempo_resolucao_dias'")
# print(f"Média de resolução (tickets resolvidos): "f"{df_tickets['tempo_resolucao_dias'].mean():.1f} dias")

# print(f"\n Shape final: {df_tickets.shape}")
# print("\nTickets de suporte arrumado")
# df_tickets.head(3)

# Arrumando a tabela Clientes

In [10]:
# Essa base não possui valores nulos. 
# Uma informação relevante é que 23 clientes se cadastraram, mas não realizaram compras. 
# Isso pode indicar que desistiram antes de comprar.

clientes_sem_pedido = df_clientes[~df_clientes["id"].isin(df_pedidos["cliente_id"])]
df_clientes["tem_pedido"] = df_clientes["id"].isin(df_pedidos["cliente_id"])

# print(f"\n{len(clientes_sem_pedido)} clientes cadastrados sem nenhum pedido")
# print(f"Segmento desses clientes:")
# print(clientes_sem_pedido["segmento"].value_counts())
# print(f"\nNova coluna: 'tem_pedido' (bool)")

# print(f"   Shape: {df_produtos.shape} | Nulos: {df_produtos.isnull().sum().sum()}")

# print("\nClientes e Produtos arrumados")
# df_clientes.head(3)

# Exportação das bases tratadas: 

In [11]:
bases_limpas = {
    "pedidos_limpo.csv":          df_pedidos,
    "clientes_limpo.csv":         df_clientes,
    "produtos_limpo.csv":         df_produtos,
    "itens_pedido_limpo.csv":     df_itens,
    "avaliacoes_limpo.csv":       df_avaliacoes,
    "tickets_suporte_limpo.csv":  df_tickets,
}

# print("\nTodos os arquivos limpos salvos em /outputs/")

caminho_outputs = Path("outputs") 

# Cria a pasta se ela não existir.
# parents=True cria toda a árvore de diretórios necessária e
# exist_ok=True evita erro se a pasta já estiver lá
caminho_outputs.mkdir(parents=True, exist_ok=True)

for nome_arquivo, df in bases_limpas.items():
    caminho = caminho_outputs / nome_arquivo
    df.to_csv(caminho, index=False, encoding="utf-8-sig")
    print(f"{nome_arquivo:<30} → {df.shape[0]:,} linhas exportadas")

pedidos_limpo.csv              → 15,000 linhas exportadas
clientes_limpo.csv             → 3,000 linhas exportadas
produtos_limpo.csv             → 200 linhas exportadas
itens_pedido_limpo.csv         → 36,740 linhas exportadas
avaliacoes_limpo.csv           → 8,000 linhas exportadas
tickets_suporte_limpo.csv      → 4,000 linhas exportadas
